In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import plotly.graph_objects as go
from scipy import ndimage
from skimage import morphology
import pymeshlab as pm
from pathlib import Path

from geometry_mnist.datagen import load_data
from geometry_mnist.paths import ROOT

def multiplot(imgs: np.ndarray, cmap: str = "gray", center: float | None = None):
    fig, axs = plt.subplots(1, imgs.shape[-1], figsize=(12, 8))
    for i in range(imgs.shape[-1]):
        if center is not None:
            norm = TwoSlopeNorm(vcenter=center)
        else:
            norm = None
        axs[i].imshow(imgs[:, :, i], cmap=cmap, norm=norm)
        axs[i].set_xticks([])
        axs[i].set_yticks([])

## Dataloading

#### from classical MNIST dataset

In [ ]:
res = 96

data, targets = load_data("mnist.h5")
imgs = data[np.random.randint(0, 10_000, size=(10, ))] # a few random samples
imgs = imgs.transpose(1, 2, 0) # cv wants h,w,c, in this case c=batch

# resize to have some more pixels to work with
imgs = cv.resize(imgs, (res, res), interpolation=cv.INTER_CUBIC) 

multiplot(imgs)

#### cleanup

some digits contain veeery small specs of white. these should be removed. but the digits can be composed of multiple disjoint components themselves, so only keeping the largest one would be wrong. additionally, the images are also binarized

In [ ]:
def clean_binarized(mask: np.ndarray, area_thresh: int) -> np.ndarray:
    # n = number of different connected componnets (background is always the first)
    # labels = basically a segmentation mask for the image; has the label number at each pixel (0 bg, 1,..., n blobs)
    # stats = ndarray, one row per label. [bb, bb, bb, bb, area]
    n, labels, stats, _ = cv.connectedComponentsWithStats(mask, connectivity=8)
    areas = stats[:, 4]
    
    # areas[labels]; numpy fancy indexing. returns the labels (img) with the respective area value placed at each pixel, corresponding to the label that was there. keep is a mask that only marks the large-blob-pixels. Areas is basically a lookup table for the mapping label -> area.
    keep = areas[labels] >= area_thresh
    
    return keep & mask

# define what a "small spec" is. about 3-5 pixels in the original mnist 28x28 -> scale to resolution used here
n_pixel_original = 4
scale = (res / 28) ** 2
small_thresh = int(n_pixel_original * scale)

# binarize and clean up
imgs_bin = np.where(imgs>100, 1, 0).astype(np.uint8)
imgs_bin_clean = np.stack([
    clean_binarized(imgs_bin[..., i], area_thresh = small_thresh) for i in range(imgs_bin.shape[-1])
], axis=-1)

multiplot(imgs_bin_clean)


## get a 2D SDF over the image

#### with morphological ops

this shows the basic idea well, but morphological ops with 3x3 kernels always shrink in a diamond pattern, so the sdf looks quite distorted

In [ ]:
# signed distance field in 2d (distance from the character boundary)
sdf = np.zeros_like(imgs_bin_clean, dtype=np.int64)

# cross erosion kernel 
kernel = np.array([
    [0, 1, 0],
    [1, 1, 1],
    [0, 1, 0],
]).astype(np.uint8)

# inwards sdf part: eroding the binarized number step by step (always by 1 pixel) and adding whats left to the sdf. his way, the further from the boundary, the larger the sdf value gets
cpy = imgs_bin_clean.copy()
for i in range(100):
    sdf = sdf - cpy
    cpy = cv.erode(cpy, kernel, iterations=1)
    if cpy.sum() == 0: break
    
# same for the outside: inverting the mask, and eroding it
cpy = 1 - imgs_bin_clean.copy()
for i in range(100):
    sdf = sdf + cpy
    cpy = cv.erode(cpy, kernel, iterations=1)
    if cpy.sum() == 0: break

multiplot(sdf - 5, cmap="PiYG", center=0)

#### with scipy distance_transform_edt

this is the cleaner version that calculate the actual euclidean distance of each "foreground" (nonzero) pixel to the nearest "background" (zero) pixel. the sdf representation also facilitate easy "thickening" of the outline, by subtracting a constant value from the sdf

In [ ]:
def edt(mask: np.ndarray) -> np.ndarray:
    out = ndimage.distance_transform_edt(mask)
    assert isinstance(out, np.ndarray)
    return out


sdf_inside = np.stack([
    edt(imgs_bin_clean[:, :, i]) for i in range(imgs_bin.shape[-1])
], axis=-1)

sdf_outside = np.stack([
    edt(1 - imgs_bin_clean[:, :, i]) for i in range(imgs_bin.shape[-1])
], axis=-1)

sdf = sdf_outside - sdf_inside # inside is defined as negative


multiplot(sdf - 5, cmap="PiYG", center=0)

## 3D SDF; from images to volume

#### get the skeletons of mnist

In [ ]:
imgs_skeleton = np.stack([
    morphology.skeletonize(imgs_bin_clean[..., i]) for i in range(imgs_bin.shape[-1])
], axis=-1)


multiplot(imgs_skeleton, cmap="grey")

#### find the closest skeleton pixel for each pixel -> give each pixel the radius of its closest skeleton pixel

(to determine stroke thickness at each pixel)

In [ ]:
def assign_r_to_each_inside_pixel(imgs_skeleton, img_sdf):

    # this finds the closest skeleton pixel (index of skeleton pixel) for all pixels 
    _, indices = ndimage.distance_transform_edt(imgs_skeleton == 0, return_indices=True)

    # then, we can use the sdf as a radius-lookup table and assign a radius to each pixel (given by the closest skeleton sdf value, negative)
    radii = -img_sdf[indices[0], indices[1]]
    
    # this only makes sense on pixels inside the sdf
    return np.where(img_sdf < 0, radii, 0)

    
imgs_radii = np.stack([
    assign_r_to_each_inside_pixel(imgs_skeleton[..., i], sdf[..., i]) for i in range(imgs_bin.shape[-1])
], axis=-1) 

multiplot(imgs_radii, cmap="Reds")


#### "smooth" the radii a bit

In [ ]:
def smin_poly(a, b, k):
    h = np.clip(0.5 + 0.5 * (b - a) / k, a_min=0, a_max=1)
    return b * (1 - h) + a * h - k * h * (1 - h)

def smooth_radii(radii, skele):
    skele_radii = radii * skele
    r_typ = np.median(skele_radii[skele_radii > 0])
    
    smoothed_radii = smin_poly(
        a = radii, # value 1
        b = 1.0 * r_typ, # value 2
        k = 0.3 * r_typ  # spread
    )

    return smoothed_radii
    
imgs_radii = np.stack([
    smooth_radii(imgs_radii[..., i], imgs_skeleton[..., i]) for i in range(imgs_bin.shape[-1])
], axis=-1)

multiplot(imgs_radii, cmap="Reds")

#### find the "height-field"

-> a value at each pixel of the 2d image that represents how far it will be "extruded" in z direction

In [ ]:
n = imgs.shape[-1]

def height_field(img_sdf: np.ndarray, radii: np.ndarray) -> np.ndarray:
    # d = distance to the outline, clamped to [0, R]: 0 outside the digit, at most R (junctions can have d > R)
    d = np.clip(-img_sdf, a_min=0, a_max=radii)
    # round tube of radius R: a pixel (R - d) away from the stroke's middle line has height sqrt(R² - (R - d)²)
    return np.sqrt(radii**2 - (radii - d)**2)

def multiplot3d(n: int, figsize=(12, 3), elev: float = 35, azim: float = -60):
    fig = plt.figure(figsize=figsize)
    # computed_zorder=False: draw in the order things are added (mpl's automatic depth sorting gets layered plots wrong)
    axs = [fig.add_subplot(1, n, i + 1, projection="3d", computed_zorder=False) for i in range(n)]
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=0)
    for ax in axs:
        ax.view_init(elev=elev, azim=azim)
        ax.set_axis_off()
    return fig, axs

imgs_ztop = np.stack([
    height_field(sdf[..., i], imgs_radii[..., i]) for i in range(n)
], axis=-1)

# image (row, col) -> plot (x, y): x = col, y flipped so the digit stands upright when seen from above
I, J = np.indices((res, res))
X, Y = J, res - 1 - I

fig, axs = multiplot3d(n)
for i, ax in enumerate(axs):
    Z = np.where(sdf[..., i] < 0, imgs_ztop[..., i], np.nan)   # hide the background
    ax.plot_surface(X, Y, Z, cmap="viridis", rstride=1, cstride=1, linewidth=0, antialiased=False)
    ax.set_box_aspect((res, res, 3 * imgs_ztop.max()), zoom=1.2)   # z exaggerated 3x to see the profile

#### build the actual SDF in 3d

intersection of "inside the tube" (|z| < z_top) and "inside the outline" (2D sdf < 0), via max. not an exact distance in 3d, but negative inside / positive outside and smooth, which is all marching cubes needs.

visualization: a vertical cut through the volume along the red line (x-z plane), black = the zero level, i.e. the later surface

In [ ]:
dz = 0.5                                    # z step in pixel units (finer than x/y, the tubes are only a few px thick)
nz = int(np.ceil(imgs_ztop.max())) + 3      # volume reaches from -nz to +nz: a few voxels margin above the highest point
z = np.arange(-nz, nz + dz, dz, dtype=np.float32)

def build_sdf3d(img_sdf: np.ndarray, img_ztop: np.ndarray, z: np.ndarray) -> np.ndarray:
    inside_tube = np.abs(z)[None, None, :] - img_ztop[:, :, None]   # < 0 between -z_top and +z_top
    inside_outline = img_sdf[:, :, None]                            # < 0 inside the digit (repeated for every z)
    return np.maximum(inside_tube, inside_outline)                  # max = intersection of both

vols = np.stack([
    build_sdf3d(sdf[..., i], imgs_ztop[..., i], z) for i in range(n)
], axis=-1)  # (H, W, Z, n)


# --- visualization: x-z cut through the volume at one image row ---
slice_row = res // 2
lim = np.abs(vols.min())
cols = np.nonzero((sdf[slice_row] < 0).any(axis=-1))[0]       # columns where any digit crosses the cut
x_lo, x_hi = cols.min() - 4, cols.max() + 4

fig, axs = plt.subplots(2, n, figsize=(12, 4.5), gridspec_kw=dict(height_ratios=[res, 2 * nz * res / (x_hi - x_lo)]))
for i in range(n):
    axs[0, i].imshow(imgs_bin_clean[..., i], cmap="gray")
    axs[0, i].axhline(slice_row, color="red", linewidth=1)

    cut = vols[slice_row, :, :, i].T                             # (Z, W): the x-z plane at that row
    axs[1, i].imshow(cut, cmap="PiYG", norm=TwoSlopeNorm(vcenter=0, vmin=-lim, vmax=lim),
                     origin="lower", extent=(-0.5, res - 0.5, -nz, nz), interpolation="bilinear")
    axs[1, i].contour(np.arange(res), z, cut, levels=[0], colors="black", linewidths=0.8)
    axs[1, i].set_xlim(x_lo, x_hi)

    for ax in axs[:, i]:
        ax.set_xticks([])
        ax.set_yticks([])

## Mesh; marching cubes on the 3D SDF

the volume gets padded with "outside" values first, so the mesh is closed even if the digit touches the volume border. vertices are moved back to image coordinates: (row, col, z) with z centered at 0

In [ ]:
from skimage.measure import marching_cubes

def mesh_from_sdf3d(vol: np.ndarray, z: np.ndarray, dz: float) -> tuple[np.ndarray, np.ndarray]:
    vol = np.pad(vol, 1, constant_values=vol.max())      # 1 voxel of "outside" all around -> closed mesh
    verts, faces, _, _ = marching_cubes(vol, level=0.0, spacing=(1.0, 1.0, dz))
    verts -= (1.0, 1.0, dz)                              # undo the padding offset
    verts[:, 2] += z[0]                                  # z index 0 -> z = -nz
    return verts, faces

meshes = [mesh_from_sdf3d(vols[..., i], z, dz) for i in range(n)]   # list of (verts, faces)


# --- visualization: mesh hovering above the original (upscaled) digit ---
z_floor = -nz - 4
fig, axs = multiplot3d(n, figsize=(16, 4), elev=25)
for i, ax in enumerate(axs):
    verts, faces = meshes[i]
    ax.contourf(X, Y, imgs[..., i], zdir="z", offset=z_floor, levels=20, cmap="gray")   # digit as a floor
    ax.plot_trisurf(verts[:, 1], res - 1 - verts[:, 0], verts[:, 2], triangles=faces,
                    color="lightsteelblue", linewidth=0, shade=True)
    ax.set_xlim(0, res)
    ax.set_ylim(0, res)
    ax.set_zlim(z_floor, nz)
    ax.set_box_aspect((res, res, nz - z_floor), zoom=1.2)

In [ ]:
# interactive look at one mesh
idx = 7
verts, faces = meshes[idx]

def arrow3d(start, vec, color: str, label: str, head: float = 6.0) -> list:
    # plotly has no 3d arrow: a line for the shaft + a cone for the head + a text marker at the tip
    start, vec = np.asarray(start, float), np.asarray(vec, float)
    tip = start + vec
    unit = vec / np.linalg.norm(vec)
    shaft_end = tip - unit * head                              # stop the line where the cone starts
    return [
        go.Scatter3d(x=[start[0], shaft_end[0]], y=[start[1], shaft_end[1]], z=[start[2], shaft_end[2]],
                     mode="lines", line=dict(color=color, width=10), showlegend=False, hoverinfo="skip"),
        go.Cone(x=[tip[0]], y=[tip[1]], z=[tip[2]], u=[unit[0]], v=[unit[1]], w=[unit[2]],
                anchor="tip", sizemode="absolute", sizeref=head,  # cone length = head (unit vector)
                colorscale=[[0, color], [1, color]], showscale=False, hoverinfo="skip"),
        go.Scatter3d(x=[tip[0]], y=[tip[1]], z=[tip[2]], mode="text", text=[label],
                     textposition="top center", textfont=dict(color=color, size=14),
                     showlegend=False, hoverinfo="skip"),
    ]

L = res / 3                         # arrow length
origin = (0, 0, 0)                  # bottom-left corner of the image, at mid height

go.Figure(
    data=[
        go.Mesh3d(
            x = verts[:, 1],              # image column -> x (points right in the image)
            y = res - 1 - verts[:, 0],    # image row, flipped -> y (points up in the image)
            z = verts[:, 2],
            i = faces[:, 0],
            j = faces[:, 1],
            k = faces[:, 2],
            color = "lightsteelblue",
            lighting = dict(ambient=0.4, diffuse=0.8, specular=0.2),
        ),
        *arrow3d(origin, (L, 0, 0), "crimson", "image right"),
        *arrow3d(origin, (0, L, 0), "seagreen", "image up"),
    ],
    layout = dict(
        scene=dict(
            aspectmode="data",
            # start looking from the front-above, with y pointing up on screen -> reads like the image
            camera=dict(up=dict(x=0, y=1, z=0), eye=dict(x=0, y=-0.6, z=3.0)),
        ),
        margin=dict(l=10, r=10, t=10, b=10),
        height=400,
    )
).show()

In [ ]:
def normalize_mesh(v: np.ndarray):
    center = (v.min(axis=0) + v.max(axis=0)) / 2 # xyz bounding box center
    v = v - center
    v = v / np.linalg.norm(v, axis=1).max() # rescale so max norm of point is 1
    
    v = v[:, [1, 0, 2]] # transpose x and y because image is defined wrong way around
    v[:, 1] = -v[:, 1] # flip y axis becaue image is the wrong way around
    
    return v

def to_meshset(v: np.ndarray, f: np.ndarray) -> mp.MeshSet:
    ms = pm.MeshSet()
    ms.add_mesh(
        pm.Mesh(
            vertex_matrix=v.astype(np.float64),
            face_matrix = f.astype(np.int32),
        ),
    )
    return ms

def save_mesh(v: np.ndarray, f: np.ndarray, path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    to_meshset(v, f).save_current_mesh(str(path))



# normalizing verts from image (ijz) to xyz coords    
verts_normed = normalize_mesh(verts)
faces_normed = faces

save_mesh(verts_normed, faces_normed, ROOT / "data" / "meshes" / f"digit_{i}_raw.ply")


# remeshing in pymeshlab
ms = to_meshset(verts_normed, faces_normed)
ms.meshing_isotropic_explicit_remeshing(iterations=10, targetlen=pm.PureValue(0.02))
ms.apply_coord_taubin_smoothing(stepsmoothnum=10)

# back to numpy
m = ms.current_mesh()
verts_remesh, faces_remesh = m.vertex_matrix(), m.face_matrix()


# plotting
go.Figure(
    data=[
        go.Mesh3d(
            x = verts_remesh[:, 0],
            y = verts_remesh[:, 1], 
            z = verts_remesh[:, 2],
            i = faces_remesh[:, 0],
            j = faces_remesh[:, 1],
            k = faces_remesh[:, 2],
            color = "lightsteelblue",
            lighting = dict(ambient=0.4, diffuse=0.8, specular=0.2),
        ),
    ],
    layout = dict(
        scene=dict(aspectmode="data"),
        margin=dict(l=10, r=10, t=10, b=10),
        height=400,
    )
).show()